# dataloader-batching — worked example 1: Verify every index appears once per epoch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-batching`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `DataLoader` over a `TensorDataset` partitions the dataset into batches each epoch. With `drop_last=False`, the union of all batches in one epoch is exactly the full dataset — every example appears in exactly one batch, regardless of whether `shuffle` is on. `shuffle` only reorders which example lands in which batch; it never drops or duplicates examples.

## Worked solution

**Goal:** confirm that one full pass over a shuffled loader recovers every original index exactly once.

1. Build a dataset of distinct integer ids `0..N-1` so each example is identifiable. We use `t.arange(N)` as both the only tensor and the natural ground-truth label set.
2. Wrap it in a `TensorDataset` and a `DataLoader(..., shuffle=True)`. We re-seed with `t.manual_seed(0)` *before* constructing the loader so the shuffle order is reproducible — the grader does not reseed for us.
3. Iterate the loader once. Each batch `b` is a 1-tuple `(ids_tensor,)` because the dataset holds one tensor; pull `b[0]` and collect its values.
4. Concatenate all collected batch tensors into one long vector with `t.cat`. Its length must equal `N` (coverage, no drops) and its sorted values must equal `0..N-1` (each id exactly once, no duplicates).
5. Sorting before comparison is the key trick: shuffle scrambles order, so we compare *sets*, not sequences. `t.sort(...).values` gives a canonical order to compare against `t.arange(N)`.

**Why it works:** `drop_last` defaults to `False`, so the final partial batch is kept; shuffling is a permutation, which is a bijection on indices, so coverage is preserved exactly.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def collect_epoch_ids(N, batch_size):
    t.manual_seed(0)
    ds = TensorDataset(t.arange(N))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    seen = []
    for (ids,) in loader:
        seen.append(ids)
    all_ids = t.cat(seen)
    return all_ids


all_ids = collect_epoch_ids(23, 5)
sorted_ids = t.sort(all_ids).values
print('count:', all_ids.numel())
print('covers 0..N-1 exactly once:', bool(t.equal(sorted_ids, t.arange(23))))